# Flock Size Modeling: Glassbox vs Blackbox

This notebook uses `output/wahis_hxnx_event_level_with_faostat.csv` to predict `last_flock_lost_total` as the flock-size target.

Approach:
- use a time-aware split based on `event_anchor_year`
- compare a glassbox model (`Ridge`) against a blackbox model (`RandomForest`)
- report holdout MAE, RMSE, and R^2

Notes:
- the target is extremely skewed, so both models train on `log1p(y)` via `TransformedTargetRegressor`
- obvious leakage is removed, especially `last_quant_total_1_deaths_killed_slaughtered_total`, which is almost identical to the target


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = ROOT / "output" / "wahis_hxnx_event_level_with_faostat.csv"
TARGET = "last_flock_lost_total"
TEST_START_YEAR = 2024


In [2]:
df = pd.read_csv(DATA_PATH)
print(f"Rows, columns: {df.shape}")
print(f"Target column: {TARGET}")
print("\nTarget summary:")
print(df[TARGET].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_string())
print("\nEvents by anchor year:")
print(df["event_anchor_year"].value_counts().sort_index().to_string())


Rows, columns: (2098, 37)
Target column: last_flock_lost_total

Target summary:
count         2,098.000
mean        353,108.961
std       4,633,722.183
min               0.000
50%             670.000
75%          35,699.000
90%         320,326.300
95%         997,851.000
99%       5,863,864.130
max     203,648,009.000

Events by anchor year:
event_anchor_year
2005     11
2006     62
2007     47
2008     34
2009     27
2010     30
2011     33
2012     23
2013     32
2014     46
2015     66
2016     92
2017    102
2018     73
2019     33
2020     55
2021    207
2022    207
2023    239
2024    244
2025    253
2026    182


In [3]:
date_cols = [
    "event_started_on",
    "event_confirmed_on",
    "event_ended_on",
    "event_created_at",
    "first_outbreak_start",
    "last_outbreak_end",
    "last_outbreak_reference",
]

for col in date_cols:
    dt = pd.to_datetime(df[col], errors="coerce", utc=True)
    df[f"{col}_month"] = dt.dt.month
    df[f"{col}_quarter"] = dt.dt.quarter

leakage_cols = {
    TARGET,
    "last_quant_total_1_deaths_killed_slaughtered_total",
    "report_numbers",
    "admin_divisions",
    "locations",
    *date_cols,
}

feature_cols = [col for col in df.columns if col not in leakage_cols]
X = df[feature_cols].copy()
y = df[TARGET].copy()

train_mask = df["event_anchor_year"] < TEST_START_YEAR
X_train = X.loc[train_mask].copy()
X_test = X.loc[~train_mask].copy()
y_train = y.loc[train_mask].copy()
y_test = y.loc[~train_mask].copy()

categorical_cols = X_train.select_dtypes(include=["object", "bool"]).columns.tolist()
numeric_cols = [col for col in X_train.columns if col not in categorical_cols]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Numeric features: {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")
print(f"Correlation between target and leaked deaths column: {df[[TARGET, 'last_quant_total_1_deaths_killed_slaughtered_total']].corr().iloc[0, 1]:.6f}")


Train shape: (1419, 39)
Test shape: (679, 39)
Numeric features: 27
Categorical features: 12
Correlation between target and leaked deaths column: 0.999996


In [4]:
ridge_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_cols,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=0.02)),
            ]),
            categorical_cols,
        ),
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            numeric_cols,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=0.02)),
            ]),
            categorical_cols,
        ),
    ]
)

models = {
    "Baseline Median": TransformedTargetRegressor(
        regressor=DummyRegressor(strategy="median"),
        func=np.log1p,
        inverse_func=np.expm1,
    ),
    "Glassbox Ridge": TransformedTargetRegressor(
        regressor=Pipeline([
            ("preprocessor", ridge_preprocessor),
            ("model", Ridge(alpha=3.0)),
        ]),
        func=np.log1p,
        inverse_func=np.expm1,
    ),
    "Blackbox RandomForest": TransformedTargetRegressor(
        regressor=Pipeline([
            ("preprocessor", tree_preprocessor),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=300,
                    min_samples_leaf=2,
                    random_state=42,
                    n_jobs=2,
                ),
            ),
        ]),
        func=np.log1p,
        inverse_func=np.expm1,
    ),
}


In [5]:
fitted_models = {}
results = []
predictions = pd.DataFrame({"actual": y_test})

for model_name, estimator in models.items():
    estimator.fit(X_train, y_train)
    preds = np.clip(estimator.predict(X_test), 0, None)
    fitted_models[model_name] = estimator
    predictions[model_name] = preds
    results.append(
        {
            "model": model_name,
            "mae": mean_absolute_error(y_test, preds),
            "rmse": root_mean_squared_error(y_test, preds),
            "r2": r2_score(y_test, preds),
        }
    )

results_df = pd.DataFrame(results).sort_values(["rmse", "mae"]).reset_index(drop=True)
results_df


,model,mae,rmse,r2
0,Glassbox Ridge,"440,362.838","6,654,162.944",0.283
1,Blackbox RandomForest,"430,135.046","7,591,765.234",0.067
2,Baseline Median,"491,748.753","7,875,833.028",-0.004


In [6]:
best_rmse_row = results_df.loc[results_df["rmse"].idxmin()]
best_mae_row = results_df.loc[results_df["mae"].idxmin()]

print(f"Best RMSE model: {best_rmse_row['model']} ({best_rmse_row['rmse']:,.3f})")
print(f"Best MAE model: {best_mae_row['model']} ({best_mae_row['mae']:,.3f})")
print("\nSample predictions:")
predictions.head(10)


Best RMSE model: Glassbox Ridge (6,654,162.944)
Best MAE model: Blackbox RandomForest (430,135.046)

Sample predictions:


,actual,Baseline Median,Glassbox Ridge,Blackbox RandomForest
1419,273.000,"1,202.000",47.699,67.052
1420,7.000,"1,202.000",36.255,42.507
1421,139.000,"1,202.000",21.383,36.812
1422,"44,911.000","1,202.000","164,994.468","26,359.813"
1423,68.000,"1,202.000",51.157,24.755
1424,227.000,"1,202.000",13.153,5.119
1425,1.000,"1,202.000",16.146,4.800
1426,351.000,"1,202.000",0.000,"1,416.676"
1427,"19,497.000","1,202.000","44,538.145","6,715.642"
1428,240.000,"1,202.000",129.198,"8,835.808"


In [7]:
ridge_pipeline = fitted_models["Glassbox Ridge"].regressor_
ridge_feature_names = ridge_pipeline.named_steps["preprocessor"].get_feature_names_out()
ridge_coefficients = ridge_pipeline.named_steps["model"].coef_

ridge_importance_df = (
    pd.DataFrame(
        {
            "feature": ridge_feature_names,
            "coefficient": ridge_coefficients,
        }
    )
    .assign(abs_coefficient=lambda frame: frame["coefficient"].abs())
    .sort_values("abs_coefficient", ascending=False)
    .head(15)
)
ridge_importance_df


,feature,coefficient,abs_coefficient
70,cat__last_quant_total_1_species_name_Birds,2.905,2.905
33,cat__disease_name_infrequent_sklearn,-2.888,2.888
55,cat__epi_unit_types_Backyard,-2.751,2.751
27,cat__disease_name_High pathogenicity avian inf...,2.095,2.095
58,cat__epi_unit_types_Farm | Backyard,2.014,2.014
29,cat__disease_name_Highly pathogenic avian infl...,1.717,1.717
60,cat__epi_unit_types_Not applicable,-1.546,1.546
56,cat__epi_unit_types_Backyard | Farm,1.535,1.535
3,num__cluster_count_total,1.509,1.509
57,cat__epi_unit_types_Farm,1.299,1.299


In [8]:
rf_pipeline = fitted_models["Blackbox RandomForest"].regressor_
rf_feature_names = rf_pipeline.named_steps["preprocessor"].get_feature_names_out()
rf_importances = rf_pipeline.named_steps["model"].feature_importances_

rf_importance_df = (
    pd.DataFrame(
        {
            "feature": rf_feature_names,
            "importance": rf_importances,
        }
    )
    .sort_values("importance", ascending=False)
    .head(15)
)
rf_importance_df


,feature,importance
70,cat__last_quant_total_1_species_name_Birds,0.480
3,num__cluster_count_total,0.101
6,num__mean_latitude,0.055
12,num__faostat_production__meat_of_chickens_fres...,0.043
5,num__mean_longitude,0.036
4,num__quant_total_entry_count_total,0.036
57,cat__epi_unit_types_Farm,0.030
9,num__outbreak_span_days,0.029
31,cat__disease_name_Influenza A viruses of high ...,0.026
2,num__outbreak_count,0.024


## Takeaway

Use the comparison table above as the main decision point.

- If you care most about explainability and the best RMSE, use the glassbox Ridge model.
- If you care most about absolute-error robustness on individual events, compare whether the RandomForest MAE is better on the holdout set.
- Neither model should be interpreted as causal; this is predictive modeling on an event-level observational table.
